## **Grupo 10: Salmonera** 

Archivo ejecutado desde Visual Studio Code

Los archivos "origen_acuicultura.sqlite", "origen_costos_alimentos.csv" y "origen_eventos_sanitarios.json" deben estar en la misma rama de este archivo.

Este código sigue las instrucciones de la ayudantía 1

## Lectura de archivos

**0. Setup**

In [ ]:
# Instalar PySpark (fijamos 3.5.3 por compatibilidad con el Java de Colab; toma ~1 minuto)
%pip install -q pyspark==3.5.3

In [ ]:
# Arrancar Spark
import sqlite3, json, requests, os
import pandas as pd
from datetime import date
from pyspark.sql import SparkSession
from pyspark.sql.functions import explode, col

spark = SparkSession.builder.appName("ayudantia-extraccion").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "listo")

**1. Archivo origen_acuicultura.sqlite**

In [ ]:
con = sqlite3.connect("origen_acuicultura.sqlite")
tablas = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", con)
print("tablas:", tablas["name"].tolist())

In [ ]:
pdf_zonas = pd.read_sql("SELECT * FROM zonas", con)   # leer con sqlite
zonas = spark.createDataFrame(pdf_zonas)              # pasar a Spark
zonas.printSchema()
print("filas:", zonas.count())
zonas.show(5)

In [ ]:
pdf_centros = pd.read_sql("SELECT * FROM centros", con)   # leer con sqlite
centros = spark.createDataFrame(pdf_centros)              # pasar a Spark
centros.printSchema()
print("filas:", centros.count())
centros.show(5)

In [ ]:
pdf_jaulas = pd.read_sql("SELECT * FROM jaulas", con)   # leer con sqlite
jaulas = spark.createDataFrame(pdf_jaulas)              # pasar a Spark
jaulas.printSchema()
print("filas:", jaulas.count())
jaulas.show(5)

In [ ]:
pdf_especies = pd.read_sql("SELECT * FROM especies", con)   # leer con sqlite
especies = spark.createDataFrame(pdf_especies)              # pasar a Spark
especies.printSchema()
print("filas:", especies.count())
especies.show(5)

In [ ]:
pdf_lotes = pd.read_sql("SELECT * FROM lotes", con)   # leer con sqlite
lotes = spark.createDataFrame(pdf_lotes)              # pasar a Spark
lotes.printSchema()
print("filas:", lotes.count())
lotes.show(42)

In [ ]:
pdf_operacion_diaria = pd.read_sql("SELECT * FROM operacion_diaria", con)   # leer con sqlite
operacion_diaria = spark.createDataFrame(pdf_operacion_diaria)              # pasar a Spark
operacion_diaria.printSchema()
print("filas:", operacion_diaria.count())
operacion_diaria.show(15)

In [ ]:
pdf_cosechas = pd.read_sql("SELECT * FROM cosechas", con)   # leer con sqlite
cosechas = spark.createDataFrame(pdf_cosechas)              # pasar a Spark
cosechas.printSchema()
print("filas:", cosechas.count())
cosechas.show(5)

**2. Origen_costos_alimento**

In [ ]:
# Lectura por defecto (utf-8): los acentos salen rotos
malo = spark.read.option("header", True).option("sep", ";").csv("origen_costos_alimento.csv")
malo.select("tipo_alimento", "mes", "precio_clp_kg", "proveedor", "moneda", "glosa").show(5, truncate=False)

In [ ]:
productos = (spark.read
             .option("header", True)
             .option("sep", ";")
             .option("encoding", "iso-8859-1")
             .csv("origen_costos_alimento.csv"))
print("filas:", productos.count(), "| columnas:", len(productos.columns))
productos.show(5, truncate=False)

**3. origen_eventos_sanitarios.json**

In [ ]:
formulas = spark.read.option("multiline", True).json("origen_eventos_sanitarios.json")
formulas.printSchema()   # fijate en el tipo anidado: array de structs, y dentro otro array

In [ ]:
eventos_jaulas = (formulas
    # 1. Explode the main array: "eventos"
    .select("anio", "empresa", explode("eventos").alias("ev"))

    # 2. Select the fields from the struct ("ev") and explode the nested array ("jaulas_afectadas")
    .select(
        "anio",
        "empresa",
        "ev.evento_id",
        "ev.centro_id",
        "ev.tipo",
        "ev.fecha_inicio",
        "ev.fecha_fin",
        "ev.mortalidad_asociada",
        "ev.tratamiento",
        "ev.jaulas_afectadas"
    )
)

print("Filas (eventos x jaula):", eventos_jaulas.count())
eventos_jaulas.show(10, truncate=False)

**4. Conexión API Dolar**

In [ ]:
resp = requests.get("https://mindicador.cl/api/dolar/2026", timeout=20)
print("codigo de respuesta:", resp.status_code)   # 200 = OK
serie = resp.json()["serie"]

dolar = spark.createDataFrame(pd.DataFrame(serie))
print("filas:", dolar.count())
dolar.show(5)

## **Limpieza de datos**

In [ ]:
from pyspark.sql.functions import col, to_date, coalesce, regexp_replace, trim, lower

In [ ]:
#dejar todo en el mismo formato

#Columnas a revisar: 
#lote: fecha_siembra, smolts_sembrados
#operacion_diaria: fecha, alimento_kg, biomasa_kg, crecimiento_kg, temperatura_agua
#cosechas: fecha, kilos_vivos, kilo_producto
#producto: tipo_alimento, mes, precio_clp_kg , precio, moneda
#eventos_jaulas: fecha_inicio, fecha_fin, mortalidad asociada
#dolar : fecha, valor

In [ ]:
def limpiar_y_reportar(df, nombre_tabla, columnas_criticas):
    total_ini = df.count()
    
    condicion_invalida = col(columnas_criticas[0]).isNull()
    for c in columnas_criticas:
        # Convertimos la columna a string temporalmente para que trim() y lower() no crasheen con números o arrays
        col_str = col(c).cast("string")
        condicion_basura = (
            col(c).isNull() | 
            (trim(col_str) == "") | 
            lower(col_str).isin([c.lower(), "sin dato", "total", "null", "nan"])
        )
        condicion_invalida = condicion_invalida | condicion_basura
        
    if "poblacion" in df.columns:
        # También aseguramos que poblacion se lea como string antes de quitarle puntos
        condicion_invalida = condicion_invalida | (regexp_replace(col("poblacion").cast("string"), "\\.", "").cast("double") <= 0)

    df_excluidos = df.filter(condicion_invalida)
    df_limpio = df.filter(~condicion_invalida)
    
    total_fin = df_limpio.count()
    excluidos_count = total_ini - total_fin
    cobertura = (total_fin / total_ini) * 100 if total_ini > 0 else 0
    
    print(f"\n--- CALIDAD DE DATOS: {nombre_tabla.upper()} ---")
    print(f"Registros iniciales : {total_ini}")
    print(f"Registros excluidos : {excluidos_count}")
    print(f"Cobertura útil      : {cobertura:.2f}%")
    
    if excluidos_count > 0:
        print(f"Muestra de filas excluidas:")
        df_excluidos.select(*columnas_criticas).show(5, truncate=False)
        
    return df_limpio, df_excluidos

In [ ]:
# Aplicamos la función a los archivos principales
zonas_limpias, zonas_excluidas = limpiar_y_reportar(
    zonas, "Zonas", ["zona_id", "zona"]
)
centros_limpios, centros_excluidos = limpiar_y_reportar(
    centros, "Centros", ["centro_id", "nombre", "zona_id"]
)
jaulas_limpios, jaulas_excluidos = limpiar_y_reportar(
    jaulas, "Jaulas", ["jaula_id", "centro_id", "capacidad"]
)
especies_limpios, especies_excluidos = limpiar_y_reportar(
    especies, "Especies", ["especie_id", "especie"]
)
lotes_limpios, lotes_excluidos = limpiar_y_reportar(
    lotes, "Lotes", ["lote_id", "centro_id", "jaula_id", "fecha_siembra", "smolts_sembrados"]
)

operacion_diaria_limpia, op_excluidos = limpiar_y_reportar(
    operacion_diaria, "Operación Diaria", ["fecha","centro_id","jaula_id","lote_id","especie_id","alimento_kg","mortalidad_n","poblacion","biomasa_kg","crecimiento_kg","temperatura_agua","zona","mes"]
)

cosechas_limpias, cosechas_excluidas = limpiar_y_reportar(
    cosechas, "Cosechas", ["cosecha_id","lote_id","centro_id","fecha","kilos_vivos","kilos_producto","calibre"]
)
productos_limpios, productos_excluidas = limpiar_y_reportar(
    productos, "Productos", ["tipo_alimento","mes","precio_clp_kg","proveedor","moneda","glosa"]
)

eventos_limpios, eventos_excluidos = limpiar_y_reportar(
    eventos_jaulas, "Eventos Sanitarios", ["anio","empresa","evento_id","centro_id","tipo","fecha_inicio","fecha_fin","mortalidad_asociada","tratamiento","jaulas_afectadas"]
)
dolar_limpio, dolar_excluido = limpiar_y_reportar(
    dolar, "Dolar", ["fecha", "valor"]
)


**Fechas**

In [ ]:
spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")

In [ ]:
spark.conf.set("spark.sql.codegen.wholeStage", "false")

In [ ]:
from pyspark.sql.functions import col, coalesce, to_date, regexp_replace, trim, lower, when, year, month, dayofmonth, make_date, lit

def limpiar_fechas(df, nombre, columnas_fecha):
    df_fmt = df
    columnas_presentes = [c for c in columnas_fecha if c in df_fmt.columns]
    
    for c in columnas_presentes:
        # 1. Limpieza base de texto
        fecha_str = trim(lower(col(c).cast("string")))
        
        meses = {
            "ene": "01", "feb": "02", "mar": "03", "abr": "04", "may": "05", "jun": "06", 
            "jul": "07", "ago": "08", "sep": "09", "oct": "10", "nov": "11", "dic": "12"
        }
        for mes_texto, mes_num in meses.items():
            fecha_str = regexp_replace(fecha_str, mes_texto, mes_num)
            
        fecha_str = regexp_replace(fecha_str, "[T\\s].*", "")
        
        # 2. Primer parseo con coalesce
        df_fmt = df_fmt.withColumn(c, coalesce(
            to_date(fecha_str, "yyyy-MM-dd"),     
            to_date(fecha_str, "dd-MM-yyyy"),     
            to_date(fecha_str, "dd/MM/yyyy"),     
            to_date(fecha_str, "dd-MM-yy"),       
            to_date(fecha_str, "dd/MM/yy")        
        ))
        
        # 3. Doble verificación (Inversión de año y día si año < 100)
        df_fmt = df_fmt.withColumn(
            c,
            when(year(col(c)) < 100,
                 make_date(dayofmonth(col(c)) + 2000, month(col(c)), year(col(c)))
            ).otherwise(col(c))
        )
        
    # 4. NUEVO: Filtro de validación (Meses inválidos y Nulos)
    if columnas_presentes:
        # Empezamos con una condición falsa para ir sumándole los errores
        condicion_invalida = lit(False)
        for c in columnas_presentes:
            # Atrapamos si la fecha colapsó a NULL (ej. mes 13) o si el mes está fuera del rango 1-12
            condicion_invalida = condicion_invalida | col(c).isNull() | ~month(col(c)).between(1, 12)
            
        df_excluidos = df_fmt.filter(condicion_invalida)
        df_limpio = df_fmt.filter(~condicion_invalida)
        
        excluidos_count = df_excluidos.count()
        
        if excluidos_count > 0:
            print(f"⚠️ Se excluyeron {excluidos_count} filas por fechas inválidas o nulas.")
            print("Muestra de registros excluidos (anotar en Bitácora):")
            df_excluidos.select(*columnas_presentes).show(5)
        else:
            print(f"✅ Todas las fechas son válidas. Cero exclusiones. Data: {nombre}")
            
        return df_limpio
    
    # Si no había ninguna columna de fecha, retornamos el DF intacto y uno vacío
    return df_fmt

In [ ]:
diccionario_dfs = {
    "zonas": zonas_limpias,
    "centros": centros_limpios,
    "jaulas": jaulas_limpios,
    "especies": especies_limpios,
    "lotes": lotes_limpios,
    "operacion_diaria": operacion_diaria_limpia,
    "cosechas": cosechas_limpias,
    "productos": productos_limpios,
    "eventos": eventos_limpios,
    "dolar": dolar_limpio
}

In [ ]:
for nombre, df in diccionario_dfs.items():
    columnas_con_fecha = [c for c in df.columns if "fecha" in c]
    if columnas_con_fecha:
        diccionario_dfs[nombre] = limpiar_fechas(df, nombre, columnas_con_fecha)

In [ ]:
zonas_limpias = diccionario_dfs["zonas"]
centros_limpios = diccionario_dfs["centros"]
jaulas_limpios = diccionario_dfs["jaulas"]
especies_limpios = diccionario_dfs["especies"]
lotes_limpios = diccionario_dfs["lotes"]
operacion_diaria_limpia = diccionario_dfs["operacion_diaria"]
cosechas_limpias = diccionario_dfs["cosechas"]
productos_limpios = diccionario_dfs["productos"]
eventos_limpios = diccionario_dfs["eventos"]
dolar_limpio = diccionario_dfs["dolar"]

In [ ]:
lotes_limpios.show(5)


In [ ]:
operacion_diaria_limpia.show(5)


In [ ]:
cosechas_limpias.show(5)


In [ ]:
productos_limpios.show(5)
total_fechas = productos_limpios.select('mes').distinct().count()
productos_limpios.select('mes').distinct().show(total_fechas)


In [ ]:
eventos_limpios.show(5)


In [ ]:
dolar_limpio.show(5)

**Numeros**

kg

In [ ]:
from pyspark.sql.functions import col, regexp_replace, trim, lit

In [ ]:
def arreglar_kg(df, nombre_tabla, columnas_kg):
    df_fmt = df
    columnas_presentes = [c for c in columnas_kg if c in df_fmt.columns]
    
    if columnas_presentes:
        # 1. Creamos una condición para encontrar nulos en las columnas especificadas
        condicion_nula = lit(False)
        for c in columnas_presentes:
            condicion_nula = condicion_nula | col(c).isNull()
            
        # 2. Separar excluidos (nulos) y limpios
        df_excluidos = df_fmt.filter(condicion_nula)
        df_limpio = df_fmt.filter(~condicion_nula)
        
        # 3. Limpiar los formatos de los números en el DataFrame limpio
        for c in columnas_presentes:
            num_str = trim(col(c).cast("string"))
            # Quitar puntos (separador de miles chileno)
            num_str = regexp_replace(num_str, "\\.", "")
            # Reemplazar comas por puntos (decimales)
            num_str = regexp_replace(num_str, ",", ".")
            # Convertir finalmente a DoubleType
            df_limpio = df_limpio.withColumn(c, num_str.cast("integer"))
            
        excluidos_count = df_excluidos.count()
        
        # 4. Reporte
        print(f"\n--- REVISIÓN DE KILOS: {nombre_tabla.upper()} ---")
        if excluidos_count > 0:
            print(f"⚠️ Se excluyeron {excluidos_count} filas por valores nulos en columnas de kilos.")
            print(f"Muestra de registros excluidos (anotar en Bitácora): en {nombre_tabla}")
            df_excluidos.select(*columnas_presentes).show(5)
        else:
            print(f"✅ Todos los datos en '{nombre_tabla}' son válidos. Cero exclusiones.")
            print("Muestra de los datos limpios:")
            df_limpio.select(*columnas_presentes).show(5)
            
        return df_limpio
    
    return df_fmt
        
    

In [ ]:
diccionario_dfs = {
    "zonas": zonas_limpias,
    "centros": centros_limpios,
    "jaulas": jaulas_limpios,
    "especies": especies_limpios,
    "lotes": lotes_limpios,
    "operacion_diaria": operacion_diaria_limpia,
    "cosechas": cosechas_limpias,
    "productos": productos_limpios,
    "eventos": eventos_limpios,
    "dolar": dolar_limpio
}

In [ ]:
for nombre, df in diccionario_dfs.items():
    columnas_con_kg = [c for c in df.columns if "precio_clp_kg" not in c.lower() and ("kg" in c.lower() or "kilos" in c.lower() or "kilo" in c.lower())]
    if columnas_con_kg:
        diccionario_dfs[nombre] = arreglar_kg(df, nombre, columnas_con_kg)

temperatura

In [ ]:
from pyspark.sql.functions import col, regexp_replace, trim, lit

def arreglar_temperatura(df, nombre_tabla, columnas_temperatura):
    df_fmt = df
    columnas_presentes = [c for c in columnas_temperatura if c in df_fmt.columns]
    
    if columnas_presentes:
        # 1. Creamos una condición para encontrar nulos
        condicion_nula = lit(False)
        for c in columnas_presentes:
            condicion_nula = condicion_nula | col(c).isNull()
            
        # 2. Separar excluidos y limpios para el reporte de calidad
        df_excluidos = df_fmt.filter(condicion_nula)
        df_limpio = df_fmt.filter(~condicion_nula)
        excluidos_count = df_excluidos.count()
        
        # 3. Limpiar el formato de temperatura
        for c in columnas_presentes:
            num_str = trim(col(c).cast("string"))
            # Solo cambiamos la coma por punto (ej. "7,6" -> "7.6")
            num_str = regexp_replace(num_str, ",", ".")
            # Convertimos a DoubleType para mantener el decimal en Tableau
            df_limpio = df_limpio.withColumn(c, num_str.cast("double"))
            
        # 4. Reporte
        print(f"\n--- REVISIÓN DE TEMPERATURA: {nombre_tabla.upper()} ---")
        if excluidos_count > 0:
            print(f"⚠️ Se excluyeron {excluidos_count} filas por valores nulos en columnas de temperatura.")
            print("Muestra de registros excluidos (anotar en Bitácora):")
            df_excluidos.select(*columnas_presentes).show(5)
        else:
            print(f"✅ Todos los datos en '{nombre_tabla}' son válidos. Cero exclusiones.")
            print("Muestra de los datos limpios:")
            df_limpio.select(*columnas_presentes).show(5)
            
        return df_limpio, df_excluidos
    
    return df_fmt, df.filter(lit(False))

In [ ]:
diccionario_dfs = {
    "zonas": zonas_limpias,
    "centros": centros_limpios,
    "jaulas": jaulas_limpios,
    "especies": especies_limpios,
    "lotes": lotes_limpios,
    "operacion_diaria": operacion_diaria_limpia,
    "cosechas": cosechas_limpias,
    "productos": productos_limpios,
    "eventos": eventos_limpios,
    "dolar": dolar_limpio
}

In [ ]:
for nombre, df in diccionario_dfs.items():
    columnas_con_t = [c for c in df.columns if "temperatura" in c]
    if columnas_con_t:
        diccionario_dfs[nombre] = arreglar_temperatura(df, nombre, columnas_con_t)

smolts

In [ ]:
from pyspark.sql.functions import col, regexp_replace, lit

def arreglar_smolts(df, nombre_tabla, columnas):
    df_fmt = df
    columnas_presentes = [c for c in columnas if c in df_fmt.columns]
    
    if columnas_presentes:
        condicion_invalida = lit(False)
        for c in columnas_presentes:
            # Extraemos temporalmente solo los dígitos para ver si queda algo
            solo_numeros = regexp_replace(col(c).cast("string"), "[^0-9]", "")
            # Es inválido si es nulo originalmente o si al quitarle el texto quedó vacío
            condicion_invalida = condicion_invalida | col(c).isNull() | (solo_numeros == "")
            
        # Separar excluidos y limpios para tu reporte de calidad
        df_excluidos = df_fmt.filter(condicion_invalida)
        df_limpio = df_fmt.filter(~condicion_invalida)
        excluidos_count = df_excluidos.count()
        
        # Aplicar la limpieza definitiva a los datos válidos
        for c in columnas_presentes:
            # Borramos todo lo que no sea número (puntos, letras, espacios)
            num_str = regexp_replace(col(c).cast("string"), "[^0-9]", "")
            # Convertimos directamente a IntegerType
            df_limpio = df_limpio.withColumn(c, num_str.cast("integer"))
            
        # Reporte por consola
        print(f"\n--- REVISIÓN DE SMOLTS/NÚMEROS ENTEROS: {nombre_tabla.upper()} ---")
        if excluidos_count > 0:
            print(f"⚠️ Se excluyeron {excluidos_count} filas por ser nulas o contener solo texto.")
            print("Muestra de registros excluidos (anotar en Bitácora):")
            df_excluidos.select(*columnas_presentes).show(5)
        else:
            print(f"✅ Todos los datos en '{nombre_tabla}' son válidos. Cero exclusiones.")
            print("Muestra de los datos limpios:")
            df_limpio.select(*columnas_presentes).show(5)
            
        return df_limpio
    
    return df_fmt


In [ ]:
diccionario_dfs = {
    "zonas": zonas_limpias,
    "centros": centros_limpios,
    "jaulas": jaulas_limpios,
    "especies": especies_limpios,
    "lotes": lotes_limpios,
    "operacion_diaria": operacion_diaria_limpia,
    "cosechas": cosechas_limpias,
    "productos": productos_limpios,
    "eventos": eventos_limpios,
    "dolar": dolar_limpio
}

In [ ]:
for nombre, df in diccionario_dfs.items():
    columnas_con_s = [c for c in df.columns if "smolts" in c]
    if columnas_con_s:
        diccionario_dfs[nombre] = arreglar_smolts(df, nombre, columnas_con_s)

precio

In [ ]:
diccionario_dfs = {
    "zonas": zonas_limpias,
    "centros": centros_limpios,
    "jaulas": jaulas_limpios,
    "especies": especies_limpios,
    "lotes": lotes_limpios,
    "operacion_diaria": operacion_diaria_limpia,
    "cosechas": cosechas_limpias,
    "productos": productos_limpios,
    "eventos": eventos_limpios,
    "dolar": dolar_limpio
}

In [ ]:
for nombre, df in diccionario_dfs.items():
    columnas_con_s = [c for c in df.columns if "smolts" in c]
    if columnas_con_s:
        diccionario_dfs[nombre] = arreglar_smolts(df, nombre, columnas_con_s)

## **Análisis**

In [ ]:
from pyspark.sql.functions import col, year, month, to_date, avg, when, lit, sum as _sum

**1. Limpieza y cobertura de datos**

**2. Preparación Costos y tiempos**

In [ ]:
productos = productos.withColumn("precio_clp", col("precio_clp_kg").cast("double"))

# Regla Principal: Precio promedio mensual único (validado por el cliente)
costo_mensual_alimento = productos.groupBy("mes").agg(
    avg("precio_clp").alias("precio_promedio_clp")
)
costo_mensual_alimento.show(5)

# Promediar el dólar por mes para simplificar el cruce (si la API trae datos diarios)
dolar_fechas = dolar.withColumn("fecha_dt", to_date(col("fecha").substr(1, 10), "yyyy-MM-dd")) \
                    .withColumn("mes", month(col("fecha_dt")))
dolar_mensual = dolar_fechas.groupBy("mes").agg(avg("valor").alias("valor_dolar_promedio"))

# Unimos Costo CLP y Dólar para tener la Dimensión de Costo final
dim_costos = costo_mensual_alimento.join(dolar_mensual, on="mes", how="left") \
    .withColumn("precio_promedio_usd", col("precio_promedio_clp") / col("valor_dolar_promedio"))

print("Dimensión de Costos Mensuales:")
dim_costos.show(5)

**3. Construcción Tabla de hechos**

In [ ]:
fct_operacion = operacion_diaria_limpia.withColumn("fecha_dt", to_date(col("fecha"), "yyyy-MM-dd")) \
                                       .withColumn("mes_cruce", month(col("fecha_dt")))

# Cálculos de negocio obligatorios validados por el cliente
fct_operacion = fct_operacion.withColumn("peso_promedio_kg", col("biomasa_kg") / col("poblacion")) \
                             .withColumn("mortalidad_kg", col("mortalidad_n") * col("peso_promedio_kg"))

# Cruzamos con la Dimensión de Costos para valorizar el alimento
fct_operacion = fct_operacion.join(dim_costos, fct_operacion.mes_cruce == dim_costos.mes, how="left") \
                             .withColumn("costo_alimento_usd", col("alimento_kg") * col("precio_promedio_usd"))

print("Tabla de Hechos Base (Operación Diaria) - Filas:", fct_operacion.count())
fct_operacion.select("fecha_dt", "centro_id", "jaula_id", "mortalidad_n", "peso_promedio_kg", "mortalidad_kg", "costo_alimento_usd").show(5)

**4. Ajustes de ventana de tiempo de eventos sanitarios**

In [ ]:
eventos_ajustados = eventos_jaulas.withColumn("fecha_inicio_adj", date_sub(to_date(col("fecha_inicio"), "yyyy-MM-dd"), 7)) \
                                  .withColumn("fecha_fin_dt", to_date(col("fecha_fin"), "yyyy-MM-dd"))

# Nos quedamos con los periodos activos únicos por jaula para evitar duplicados en el cruce
eventos_activos = eventos_ajustados.select("jaula_id", "evento_id", "fecha_inicio_adj", "fecha_fin_dt").distinct()


**9. Cruce de Operación Diaria con Eventos**

In [ ]:
fct_cruzada = fct_operacion.join(
    eventos_activos,
    (fct_operacion.jaula_id == eventos_activos.jaula_id) &
    (fct_operacion.fecha_dt >= eventos_activos.fecha_inicio_adj) &
    (fct_operacion.fecha_dt <= eventos_activos.fecha_fin_dt),
    "left"
)

# Creamos una bandera 'con_evento' (1 si hubo evento, 0 si fue un día normal)
fct_cruzada = fct_cruzada.withColumn("con_evento", when(col("evento_id").isNotNull(), 1).otherwise(0))

**# 10. Cálculo de Tasa de Mortalidad Basal**

In [ ]:
fct_cruzada = fct_cruzada.withColumn(
    "etapa_peso",
    when(col("peso_promedio_kg") < 1.5, "1. Smolt/Engorda Inicial")
    .when(col("peso_promedio_kg") <= 4.0, "2. Engorda Media")
    .otherwise("3. Cosecha/Pigmentado")
)

# Calculamos la tasa basal usando EXCLUSIVAMENTE los días sin evento declarado (con_evento == 0)
basal_df = fct_cruzada.filter(col("con_evento") == 0) \
    .groupBy("etapa_peso") \
    .agg((_sum("mortalidad_n") / _sum("poblacion")).alias("tasa_mortalidad_basal"))

print("--- Tasa de Mortalidad Basal por Etapa ---")
basal_df.show(truncate=False)

**11. Cálculo de Exceso de Mortalidad (El Hallazgo)**

In [ ]:
fct_final = fct_cruzada.join(basal_df, on="etapa_peso", how="left")

# Calculamos cuántos peces debieron morir "normalmente" ese día
fct_final = fct_final.withColumn(
    "mortalidad_basal_n", 
    (col("poblacion") * col("tasa_mortalidad_basal")).cast("int")
)

# Aislamos los peces extra que murieron por culpa del evento (solo en días con evento)
fct_final = fct_final.withColumn(
    "exceso_mortalidad_n",
    when((col("con_evento") == 1) & (col("mortalidad_n") > col("mortalidad_basal_n")), 
         col("mortalidad_n") - col("mortalidad_basal_n")
    ).otherwise(0)
)

# Valorizamos ese exceso en kilos usando el peso promedio de la jaula ese día
fct_final = fct_final.withColumn(
    "exceso_mortalidad_kg",
    col("exceso_mortalidad_n") * col("peso_promedio_kg")
)

print("--- Muestra Final: Días con Evento Sanitario Activo ---")
fct_final.filter(col("con_evento") == 1) \
         .select("fecha_dt", "jaula_id", "etapa_peso", "mortalidad_n", "mortalidad_basal_n", "exceso_mortalidad_kg") \
         .show(10)